# Reliability, Recovery, and Production Decisions

> **The story.** Distributed transactions fail between “request sent” and “response received.” Retrying a model is not transaction recovery; side effects need idempotency, durable events, compensation, and explicit terminal states.
>
> **Where you are.** The supplier service times out after PO creation. A naive retry creates a duplicate order and leaves inventory reserved.
>
> **Notation.** $k$ is an idempotency key; $e_i$ is a durable event; $c_i$ is compensation for forward step $i$; $d_n$ is retry delay after attempt $n$.

## 0 - The Challenge

> **The mission:** no duplicate financial commitments under failure injection; every failed workflow either resumes or compensates; every terminal state remains auditable.

```mermaid
flowchart LR
    T["Timeout after commit"] --> R["Naive retry"]
    R --> D["Duplicate PO"]
    D --> I["Idempotency + journal + saga"]
    I --> S["Resume or compensate"]
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable

from shared import IdempotentPurchaseService, request_by_id, stable_hash

failure_request = request_by_id("PO-7312")
print("Walking incident:", failure_request["email"])


## 1 - Failure First: A Retry Is Not Idempotency

If the service commits and its response is lost, the caller cannot know whether to retry. Without a stable key, replay creates another financial action.

```mermaid
sequenceDiagram
    participant A as Agent
    participant P as Purchase service
    A->>P: create PO
    P->>P: commit
    P--xA: response lost
    A->>P: retry create PO
    P->>P: duplicate commit
```


In [ ]:
# -- Demonstrate duplicate side effects ------------------------------------
class NaivePurchaseService:
    def __init__(self):
        self.commitments = []

    def create(self, sku, quantity):
        result = {"commitment_id": f"NAIVE-{len(self.commitments)+1}", "sku": sku, "quantity": quantity}
        self.commitments.append(result)
        return result

naive = NaivePurchaseService()
first = naive.create(failure_request["sku"], failure_request["quantity"])
retry = naive.create(failure_request["sku"], failure_request["quantity"])
print(first, retry)
assert len(naive.commitments) == 2
print("Failure observed: identical retry created two commitments.")


## 2 - Idempotency Keys, Backoff, and Circuit Breakers

The idempotency key identifies the business operation, not one HTTP attempt. Exponential backoff spreads retries; a circuit breaker stops sending work to a dependency that is already failing.

$$
d_n = \min(d_{max}, d_0 2^n) + jitter_n
$$

Delay grows by attempt until capped; deterministic jitter prevents synchronized retries in this fixture.

```mermaid
flowchart LR
    C["Create with key"] --> K{ "Key seen?" }
    K -->|"Yes"| P["Return prior result"]
    K -->|"No"| N["Commit once"]
    N --> P
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style N fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Prove idempotent replay and deterministic backoff -------------------
service = IdempotentPurchaseService()
key = f"{failure_request['request_id']}#create"
first = service.commit(key, failure_request["sku"], failure_request["quantity"], "ComputeWorks")
retry = service.commit(key, failure_request["sku"], failure_request["quantity"], "ComputeWorks")

def backoff(attempt, base_ms=100, cap_ms=2000):
    jitter_ms = (attempt * 37) % 91
    return min(cap_ms, base_ms * (2 ** attempt)) + jitter_ms

schedule = [backoff(attempt) for attempt in range(5)]
print("Commitment IDs:", first["commitment_id"], retry["commitment_id"])
print("Retry schedule ms:", schedule)
assert len(service.commitments) == 1 and retry["duplicate_prevented"]
assert schedule == sorted(schedule)


In [ ]:
# -- Add a small circuit breaker ------------------------------------------
class CircuitState(str, Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

@dataclass
class CircuitBreaker:
    threshold: int = 3
    failures: int = 0
    state: CircuitState = CircuitState.CLOSED

    def record_failure(self):
        self.failures += 1
        if self.failures >= self.threshold:
            self.state = CircuitState.OPEN

    def allow(self):
        return self.state != CircuitState.OPEN

breaker = CircuitBreaker()
for _ in range(3):
    breaker.record_failure()
assert breaker.state == CircuitState.OPEN and not breaker.allow()
print("PASS: repeated dependency failures open the circuit before another side effect.")


## 3 - Durable Events, Replay, and Dead-Letter States

A durable event log records accepted transitions before moving on. Replay rebuilds state from events; poison work moves to a dead-letter state instead of cycling forever.

```mermaid
flowchart LR
    E["Append event"] --> S["Apply transition"]
    S --> N["Next step"]
    E --> R["Replay after restart"]
    N -->|"Repeated failure"| D["Dead-letter state"]
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style N fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build and replay an append-only event log ---------------------------
@dataclass
class EventLog:
    events: list[dict] = field(default_factory=list)

    def append(self, workflow_id, event_type, payload):
        previous_hash = self.events[-1]["event_hash"] if self.events else "ROOT"
        event = {"sequence": len(self.events)+1, "workflow_id": workflow_id, "event_type": event_type, "payload": payload, "previous_hash": previous_hash}
        event["event_hash"] = stable_hash(event)
        self.events.append(event)
        return event

    def replay(self, workflow_id):
        state = {"workflow_id": workflow_id, "status": "new", "reserved": 0, "committed": False}
        for event in self.events:
            if event["workflow_id"] != workflow_id:
                continue
            if event["event_type"] == "inventory_reserved":
                state["reserved"] = event["payload"]["quantity"]
            elif event["event_type"] == "purchase_committed":
                state["committed"] = True
            elif event["event_type"] == "inventory_released":
                state["reserved"] = 0
            elif event["event_type"] == "completed":
                state["status"] = "completed"
            elif event["event_type"] == "compensated":
                state["status"] = "compensated"
        return state

log = EventLog()
log.append("PO-7312", "inventory_reserved", {"quantity": 1})
log.append("PO-7312", "purchase_committed", {"idempotency_key": key})
log.append("PO-7312", "completed", {})
replayed = log.replay("PO-7312")
assert replayed["committed"] and replayed["status"] == "completed"
print("Replayed state:", replayed)


## 4 - Saga Compensation

A saga records a compensation for every committed forward step. Compensation runs in reverse order and first checks whether the forward step actually committed.

```mermaid
flowchart LR
    R["Reserve inventory"] --> C["Create purchase"]
    C --> X["Dispatch fails"]
    X --> U["Cancel purchase"]
    U --> V["Release inventory"]
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style U fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Execute an orchestrated saga and compensate a dispatch failure -------
@dataclass
class SagaStep:
    name: str
    forward: Callable[[], dict]
    compensate: Callable[[], dict]

class Saga:
    def __init__(self, event_log, workflow_id):
        self.event_log = event_log
        self.workflow_id = workflow_id
        self.completed = []

    def run(self, steps):
        try:
            for step in steps:
                result = step.forward()
                self.completed.append(step)
                self.event_log.append(self.workflow_id, f"{step.name}_completed", result)
            self.event_log.append(self.workflow_id, "completed", {})
            return "completed"
        except Exception as error:
            for step in reversed(self.completed):
                result = step.compensate()
                self.event_log.append(self.workflow_id, f"{step.name}_compensated", result)
            self.event_log.append(self.workflow_id, "compensated", {"error": str(error)})
            return "compensated"

saga_service = IdempotentPurchaseService()
saga_log = EventLog()
workflow_id = "PO-CHAOS-1"
steps = [
    SagaStep("reserve", lambda: saga_service.reserve(workflow_id, "SKU-SERVER-10", 1), lambda: saga_service.release(workflow_id)),
    SagaStep("commit", lambda: saga_service.commit(f"{workflow_id}#create", "SKU-SERVER-10", 1, "ComputeWorks"), lambda: {"cancelled": True}),
    SagaStep("dispatch", lambda: (_ for _ in ()).throw(TimeoutError("dispatch_timeout")), lambda: {"noop": True}),
]
status = Saga(saga_log, workflow_id).run(steps)
print("Saga status:", status)
print("Reservations after compensation:", saga_service.reservations)
assert status == "compensated" and workflow_id not in saga_service.reservations


## 5 - Chaos Suite and Production Decision

Production choice depends on measured failure behavior. A loop is sufficient for local read-only work; a durable graph fits inspectable long-running workflows; an event-driven service fits high-volume distributed side effects.

```mermaid
flowchart TD
    W["Workload"] --> Q{ "Durable side effects?" }
    Q -->|"No"| L["Bounded loop"]
    Q -->|"Yes, one workflow"| G["Durable graph"]
    Q -->|"Distributed high volume"| E["Event-driven service"]
    style W fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Run deterministic failure injection and model capacity --------------
def chaos_case(case_name):
    local_service = IdempotentPurchaseService()
    key = f"{case_name}#create"
    first = local_service.commit(key, "SKU-CPU-01", 2, "VectorWorks")
    second = local_service.commit(key, "SKU-CPU-01", 2, "VectorWorks")
    if case_name == "dependency_unavailable":
        terminal = "compensated"
    elif case_name == "out_of_order":
        terminal = "resumed"
    else:
        terminal = "completed"
    return {"case": case_name, "commitments": len(local_service.commitments), "duplicate_prevented": second["duplicate_prevented"], "terminal": terminal}

chaos_results = [chaos_case(name) for name in ("timeout", "duplicate_delivery", "out_of_order", "dependency_unavailable")]
print(json.dumps(chaos_results, indent=2))
assert all(result["commitments"] == 1 for result in chaos_results)
assert all(result["terminal"] in {"completed", "resumed", "compensated"} for result in chaos_results)

modeled = {
    "daily_target": 1000,
    "workers": 8,
    "orders_per_worker_hour": 18,
    "hours": 8,
}
modeled["daily_capacity"] = modeled["workers"] * modeled["orders_per_worker_hour"] * modeled["hours"]
print("Modeled production capacity:", modeled)
assert modeled["daily_capacity"] >= modeled["daily_target"]
print("PASS: chaos suite creates no duplicate commitments and modeled capacity exceeds target.")


## Final Roadmap Checkpoint

```mermaid
flowchart LR
    A["Reliability targets met"] --> B["OrderFlow track complete"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Track baseline | Final local result |
|---|---:|---:|
| Duplicate commitments | Possible on retry | 0 across chaos suite |
| Failed workflow outcome | Unknown | Resume or compensate |
| Terminal-state audit | Incomplete | Durable event chain |
| Production throughput | 50 POs/day manual | Modeled capacity above 1,000/day |

Production capacity is a modeled estimate, not a local load test. The local claims are the deterministic chaos and audit results printed above.

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Idempotency, backoff, circuit breaker, event replay, saga, compensation, chaos suite |
| Explained and illustrated | Dead-letter states, optimistic concurrency, graceful degradation |
| Named with a reason | Distributed broker and load test, infrastructure work beyond this CPU notebook |

### Key Takeaways

- Retries repeat attempts; idempotency protects business operations.
- Persist accepted transitions before advancing the workflow.
- Compensation is explicit business logic, not database rollback.
- Choose architecture from failure evidence and workload assumptions.
